In [9]:
from pyprojroot import here
PROJECT_ROOT = here()
DATASET_DIR = PROJECT_ROOT/"Dataset"
MODEL_DIR = PROJECT_ROOT/"models"
IR_DIR = PROJECT_ROOT/"Pz and Sz"
FIG_DIR = PROJECT_ROOT/"figs"
print(PROJECT_ROOT)
print(DATASET_DIR)
print(MODEL_DIR)
print(FIG_DIR)
FIG_DIR.mkdir(parents=True, exist_ok=True)

/home/rkobayashi/anc/GFANC-custom
/home/rkobayashi/anc/GFANC-custom/Dataset
/home/rkobayashi/anc/GFANC-custom/models
/home/rkobayashi/anc/GFANC-custom/figs


In [10]:

import matplotlib.pyplot as plt
import numpy as np
import torch

from gfanc.common.loading_real_wave_noise import loading_real_wave_noise
from gfanc.common.Reading_path_test import loading_paths_from_MAT
from gfanc.data.Disturbance_generation import Disturbance_generation_from_real_noise
from gfanc.fxlms.FxLMS_algorithm_v2 import FxLMS, train_fxlms_algorithm
from gfanc.gfanc.Control_filter_selection_pre_now import Control_filter_selection_pre_now
from gfanc.gfanc.Fixed_filter_noise_cancellation_subfilters import Fixed_filter_controller

print(torch.cuda.is_available())

True


In [11]:
# Loading real noises
mdict = {}
fs = 16000
StepSize = 0.0001
sound_name = 'Helicopter'
waveform, resample_rate = loading_real_wave_noise(folde_name=PROJECT_ROOT/'Real Noise Examples/', sound_name=sound_name+'.wav')
waveform = waveform[0].unsqueeze(0)

In [ ]:
# Loading path
subfolder = "Dongyuan"
Pri_path, Secon_path = loading_paths_from_MAT(folder=IR_DIR, subfolder=subfolder, Pri_path_file_name='Primary_path.mat', Sec_path_file_name='Secondary_path.mat')
Dis, Fx, Re = Disturbance_generation_from_real_noise(fs=fs, Repet=0, wave_form=waveform, Pri_path=Pri_path, Sec_path=Secon_path)
# Dis: disturbance (cotrolled noise)， Fx: fixed-x signal, Re: repeated waveform (primary_noise) Repetition=Repet+1

In [13]:
print(waveform.shape)
print(Dis.shape)
print(Fx.shape)
print(Re.shape)

import matplotlib as mpl

mpl.rcParams['agg.path.chunksize'] = 10000

torch.Size([1, 160000])
torch.Size([160000])
torch.Size([160000])
torch.Size([160000])


In [14]:
def additional_noise(signal, snr_db):
    signal_power = signal.square().mean()
    additional_noise = torch.randn_like(signal, dtype=signal_power.dtype, device=signal_power.device)
    snr = 10 ** (snr_db / 10)
    noise_power = signal_power / snr
    print(signal_power)
    print(noise_power)
    return signal + noise_power.sqrt() * additional_noise

In [17]:
# Combine SFANC with FxNLMS
from gfanc.sfanc_fxlms.Combine_SFANC_with_FxNLMS import SFANC_FxNLMS
from gfanc.sfanc_fxlms.Control_filter_selectionSFANC import Control_filter_selectionSFANC

# prediction index
id_vector = Control_filter_selectionSFANC(fs=16000, Primary_noise=Re.unsqueeze(0)) # Primary_noise: torch.Size([1, XX])
print(id_vector)

# Using prediction index in SFANC_FxNLMS
FILE_NAME_PATH = 'models/SFANC_Pretrained_Control_filters.mat'
SFANC_FxNLMS_Cancellation = SFANC_FxNLMS(MAT_FILE=FILE_NAME_PATH, fs=16000)
Error_SFANC_FxNLMS = SFANC_FxNLMS_Cancellation.noise_cancellation(Dis=Dis, Fx=Fx, filter_index=id_vector, Stepsize=0)

plt.title('The SFANC Algorithm')
plt.plot(Time, Dis, color='blue', label='ANC off')
plt.plot(Time, Error_SFANC_FxNLMS, color='red', label='ANC on')
plt.ylabel('Magnitude')
plt.xlabel('Time (seconds)')
plt.legend()
plt.grid()
plt.savefig(sound_name + '/SFANC.png', dpi=600, bbox_inches='tight', pad_inches=0)
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'Trained models/model.pth'